<a href="https://colab.research.google.com/github/lynkvu2901/Customer-Life-Time-Value/blob/main/mini_chatbot_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Chatbot


## 1. Cài thư viện

In [1]:
!pip install -q transformers accelerate torch

## 2. Kiểm tra GPU

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Đang chạy trên:", device)

Đang chạy trên: cuda


## 3. Tải model (mã nguồn mở, nhẹ, chạy tốt trên GPU free)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Đã tải model xong!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã tải model xong!


## 4. Hàm chat (giữ lịch sử hội thoại)

- `chat_history`: lưu lại các lượt hỏi-đáp để model có "ngữ cảnh"
- `apply_chat_template`: định dạng hội thoại đúng chuẩn model được huấn luyện để hiểu vai trò user/assistant
- `system_prompt`: định hình "tính cách" và giới hạn phạm vi trả lời của bot

In [4]:
system_prompt = (
    "Bạn là một trợ lý ảo thân thiện, trả lời ngắn gọn, rõ ràng, "
    "bằng tiếng Việt trừ khi được hỏi bằng ngôn ngữ khác."
)

chat_history = [{"role": "system", "content": system_prompt}]

def chat(user_message, max_new_tokens=300):
    chat_history.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        chat_history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(response_ids, skip_special_tokens=True)

    chat_history.append({"role": "assistant", "content": reply})
    return reply

## 5. Thử chat

In [5]:
print(chat("Chào bạn, bạn có thể làm gì?"))

Tôi có thể giúp bạn với nhiều câu hỏi và thông tin. Bạn cần tôi giúp gì hôm nay?


In [6]:
print(chat("Cho mình 3 mẹo học tập trung hiệu quả hơn."))

Tất nhiên! Dưới đây là ba mẹo để giúp bạn học tập tốt hơn:

1. **Học theo nhóm**: Học với người khác có thể giúp bạn tìm thấy các phương pháp học tập mới, cũng như tạo động lực cho cả hai bạn.

2. **Kết hợp việc học và nghỉ ngơi**: Khi bạn đã học hết một bài, hãy dành thời gian nghỉ ngơi hoặc chơi game nhẹ nhàng để thư giãn não bộ.

3. **Đặt mục tiêu cụ thể**: Chia nhỏ nhiệm vụ lớn thành những mục tiêu nhỏ hơn dễ dàng thực hiện hơn. Điều này sẽ giúp bạn cảm thấy mình đang tiến gần hơn đến việc hoàn thành công việc.


## 6. Vòng lặp chat trực tiếp

In [7]:
while True:
    msg = input("Bạn: ")
    if msg.strip().lower() == "exit":
        break
    reply = chat(msg)
    print("Bot:", reply)

Bạn: hello
Bot: Hello! How can I assist you today?
Bạn: who is the richest person in the world
Bot: As of my last update in October 2023, Jeff Bezos from Amazon and Bill Gates from Microsoft are considered to be among the wealthiest people globally. However, this information can change over time as wealth changes hands and new billionaires emerge.
Bạn: wow
Bot: Thank you! If you have any more questions or need further assistance, feel free to ask.
Bạn: how many people in the world
Bot: The exact number of people on Earth is difficult to determine precisely because it's a living population that constantly grows and decreases. According to the United Nations, there were approximately 7.9 billion people living on Earth at the end of 2022. This figure may vary slightly depending on how different sources calculate global populations.
Bạn: exit
